In [29]:
from async_graph_bench.stores import CSVDataStore, DiskCacheStore
import math
import re
import pandas as pd

WHITESPACE_TOKENS_REGEX = re.compile(r"[Ġ▁Ċ▂▃\s]")

def decode(text: str):
    return text.decode("utf-8", errors="ignore") if isinstance(text, bytes) else text

def remove_whitespace(text: str) -> str:
    return WHITESPACE_TOKENS_REGEX.sub("", text)

def get_token_probabilities(alternatives, tokens):
    cleaned_alternatives = [[remove_whitespace(decode(token)), math.exp(log_prob)] for token, log_prob in alternatives]
    result = []
    for t in tokens:
        probabilities_for_token = [probability for token, probability in cleaned_alternatives if token == t]
        result.append(max(probabilities_for_token) if len(probabilities_for_token) != 0 else 0)

    return result

In [35]:
def extract_probs(col_value):
    yes_prob, no_prob = get_token_probabilities(col_value[0], ["Yes", "No"])
    return pd.Series({
        "yes_prob": yes_prob,
        "no_prob": no_prob,
        "is_yes": yes_prob > no_prob
    })

dfs = {}
for m in ["Llama-3.3-70B-Instruct","Ministral-8B-Instruct-2410","Qwen3-30B-A3B-Instruct-2507"]:
    store = DiskCacheStore("", f'{m}-MCQAAPriCoTResponseGeneratorPost')
    df = store.to_dataframe(properties=['conclusion_tokens_decoded_alternatives'])
    df[["yes_prob", "no_prob", "is_yes"]] = df["conclusion_tokens_decoded_alternatives"].apply(extract_probs)
    df.drop(columns=["conclusion_tokens_decoded_alternatives"])
    dfs[m] = df

In [36]:
# collect only the is_yes columns, one per model
is_yes_df = pd.concat(
    {model: df["is_yes"] for model, df in dfs.items()},
    axis=1
)

# count yes / no votes per row
agreement_df = pd.DataFrame(index=is_yes_df.index)
agreement_df["is_yes"] = is_yes_df.sum(axis=1)          # True counts as 1
agreement_df["is_no"] = (~is_yes_df).sum(axis=1)

In [38]:
summary = {
    "all_3_agree": ((agreement_df["is_yes"] == 3) | (agreement_df["is_no"] == 3)).sum(),
    "two_vs_one": ((agreement_df["is_yes"] == 2) | (agreement_df["is_no"] == 2)).sum(),
    "all_3_yes": (agreement_df["is_yes"] == 3).sum(),
    "all_3_no": (agreement_df["is_no"] == 3).sum(),
}
summary

{'all_3_agree': np.int64(9774),
 'two_vs_one': np.int64(226),
 'all_3_yes': np.int64(2822),
 'all_3_no': np.int64(6952)}